check first

In [9]:
import chess
import chess.pgn
import io

moves = ['e2e4,f7f6,d1h5,g7g6,h5g6']
moves_list = moves[0].split(",")

board = chess.Board()

forced_moves = []
for move_idx, move in enumerate(moves_list, start=1):
    legal_count = board.legal_moves.count()
    print(f"Move {move_idx}: {move}, Legal moves = {legal_count}")
    if legal_count == 1:
        forced_moves.append(move_idx)
    board.push_uci(move)

print("强制走子步号列表:", forced_moves)

Move 1: e2e4, Legal moves = 20
Move 2: f7f6, Legal moves = 20
Move 3: d1h5, Legal moves = 30
Move 4: g7g6, Legal moves = 1
Move 5: h5g6, Legal moves = 43
强制走子步号列表: [4]


In [8]:
import sqlite3
import pandas as pd
import chess
from tqdm import tqdm

conn = sqlite3.connect(r"C:\sqlite3\chess.db")  
query = "SELECT rowid, Moves FROM games"
df = pd.read_sql_query(query, conn)
conn.close()

#返回强制走子步号列表，同时区分没有步子和有步子但无强制走子
def get_forced_moves_from_moves(move_string):
    if move_string is None:
        return 0  # 没有步子
    
    moves_list = [m.strip() for m in move_string.split(",") if m.strip()]
    if not moves_list:
        return 0

    board = chess.Board()
    forced_moves = []
    
    for move_idx, move in enumerate(moves_list, start=1):
        legal_count = board.legal_moves.count()
        if legal_count == 1:
            forced_moves.append(move_idx)
        try:
            board.push_uci(move)
        except ValueError:
            print(f"非法走法 {move}, 跳过")
            break
    
    if not forced_moves:
        return []  # 有步子但没有强制走子
    return forced_moves

results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc='Processing games'):
    game_id = row['rowid']
    moves = row['Moves']
    forced_moves = get_forced_moves_from_moves(moves)
    results.append({'Game_ID': game_id, 'Forced_Move_Idx_List': forced_moves})

df_forced = pd.DataFrame(results)
df_forced.to_csv(r"C:\Users\Administrator\Desktop\Chess\data\input\forced_moves_per_game.csv", index=False)
print("处理完成，CSV 文件已生成")
print(df_forced.head(50))

Processing games: 100%|██████████| 1017708/1017708 [52:07<00:00, 325.37it/s] 


处理完成，CSV 文件已生成
    Game_ID Forced_Move_Idx_List
0         1                   []
1         2                   []
2         3                   []
3         4                   []
4         5                   []
5         6                   []
6         7                    0
7         8                 [70]
8         9                   []
9        10                   []
10       11                   []
11       12                   []
12       13                   []
13       14                   []
14       15             [46, 48]
15       16                   []
16       17                   []
17       18                 [52]
18       19                   []
19       20                   []
20       21                   []
21       22                 [54]
22       23                   []
23       24                   []
24       25                   []
25       26                   []
26       27                   []
27       28                   []
28       29                 

In [9]:
total_forced_moves = 0

for fm in df_forced['Forced_Move_Idx_List']:
    if isinstance(fm, list):
        total_forced_moves += len(fm)  # 只统计列表里的步数
    # 0 和其他非列表情况不计

print("总的强制走子步数:", total_forced_moves)

总的强制走子步数: 649964
